# 10K VAE Architecture Comparison Baseline

This notebook implements the baseline VAE for the 10K architecture experiment, comparing against the existing 10K Autoencoder.


In [ ]:
import os
import random
import hashlib
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 1. Load Dataset
Matching exactly the 10K AE preprocessing.


In [ ]:
!pip install -q datasets
from datasets import load_dataset
dataset = load_dataset("evilsocket/alucard-sprites")
print(dataset)


In [ ]:
def image_hash(image):
    pixels = np.asarray(image.convert("RGBA"), dtype=np.uint8)
    return hashlib.sha256(pixels.tobytes()).hexdigest()

all_hashes = []
seen_hashes = set()

for i, item in enumerate(dataset["train"]):
    h = image_hash(item["image"])
    if h not in seen_hashes:
        seen_hashes.add(h)
        all_hashes.append(i)

unique_dataset = dataset["train"].select(all_hashes)
print("Unique images:", len(unique_dataset))


## 2. Deterministic Split
Reusing Seed=42 to guarantee fair comparison with 10K AE.


In [ ]:
SEED = 42
unique_dataset = unique_dataset.shuffle(seed=SEED)

TRAIN_SIZE = 9_000
VAL_SIZE = 1_000
TEST_SIZE = 1_000
TOTAL_SIZE = TRAIN_SIZE + VAL_SIZE + TEST_SIZE

clean_dataset = unique_dataset.select(range(TOTAL_SIZE))

train_dataset = clean_dataset.select(range(0, TRAIN_SIZE))
val_dataset = clean_dataset.select(range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE))
test_dataset = clean_dataset.select(range(TRAIN_SIZE + VAL_SIZE, TOTAL_SIZE))

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))


In [ ]:
def data_generator(hf_dataset):
    for item in hf_dataset:
        image = item["image"]
        image = image.convert("RGBA")
        image_np = np.array(image, dtype=np.float32) / 255.0
        yield (image_np, image_np)

BATCH_SIZE = 32

def create_tf_dataset(hf_dataset):
    return tf.data.Dataset.from_generator(
        lambda: data_generator(hf_dataset),
        output_signature=(
            tf.TensorSpec(shape=(128, 128, 4), dtype=tf.float32),
            tf.TensorSpec(shape=(128, 128, 4), dtype=tf.float32)
        )
    )

train_tf = create_tf_dataset(train_dataset).batch(BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)
val_tf = create_tf_dataset(val_dataset).batch(BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)
test_tf = create_tf_dataset(test_dataset).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# For evaluation
test_originals = []
for images, _ in test_tf:
    test_originals.append(images.numpy())
test_originals = np.concatenate(test_originals, axis=0)
print("Pipelines ready.")


## 3. VAE Architecture
Adding probabilistic latent representations.


In [ ]:
class Sampling(layers.Layer):
    """Uses (z_mean, z_log_var) to sample z."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon


In [ ]:
latent_dim = 256

# Encoder
encoder_inputs = keras.Input(shape=(128, 128, 4))
x = layers.Conv2D(32, 3, activation="relu", padding="same", strides=2)(encoder_inputs)
x = layers.Conv2D(64, 3, activation="relu", padding="same", strides=2)(x)
x = layers.Conv2D(128, 3, activation="relu", padding="same", strides=2)(x)
x = layers.Conv2D(256, 3, activation="relu", padding="same", strides=2)(x)
x = layers.Flatten()(x)
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")
encoder.summary()

# Decoder
latent_inputs = keras.Input(shape=(latent_dim,))
x = layers.Dense(8 * 8 * 256, activation="relu")(latent_inputs)
x = layers.Reshape((8, 8, 256))(x)
x = layers.Conv2DTranspose(128, 4, activation="relu", padding="same", strides=2)(x)
x = layers.Conv2DTranspose(64, 4, activation="relu", padding="same", strides=2)(x)
x = layers.Conv2DTranspose(32, 4, activation="relu", padding="same", strides=2)(x)
decoder_outputs = layers.Conv2DTranspose(4, 4, activation="sigmoid", padding="same", strides=2)(x)
decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")
decoder.summary()


In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = keras.metrics.Mean(name="loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def call(self, inputs):
        _, _, z = self.encoder(inputs)
        return self.decoder(z)

    def train_step(self, data):
        x, _ = data
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(x)
            reconstruction = self.decoder(z)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    tf.square(x - reconstruction), axis=(1, 2, 3)
                )
            )
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            total_loss = reconstruction_loss + kl_loss
            
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def test_step(self, data):
        x, _ = data
        z_mean, z_log_var, z = self.encoder(x)
        reconstruction = self.decoder(z)
        reconstruction_loss = tf.reduce_mean(
            tf.reduce_sum(
                tf.square(x - reconstruction), axis=(1, 2, 3)
            )
        )
        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
        total_loss = reconstruction_loss + kl_loss
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3))


## 4. Train Model


In [ ]:
MODEL_DIR = "../models/10K/VAE"
os.makedirs(MODEL_DIR, exist_ok=True)
import json

class SaveHistoryCallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        # Append history safely
        hist_path = os.path.join(MODEL_DIR, "history.json")
        if os.path.exists(hist_path):
            with open(hist_path, "r") as f:
                history = json.load(f)
        else:
            history = {}
        for k, v in logs.items():
            history.setdefault(k, []).append(float(v))
        with open(hist_path, "w") as f:
            json.dump(history, f)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(filepath=os.path.join(MODEL_DIR, "latest.weights.h5", save_weights_only=True), save_weights_only=True),
    keras.callbacks.ModelCheckpoint(filepath=os.path.join(MODEL_DIR, "best.weights.h5", save_weights_only=True), save_best_only=True, monitor="val_loss", save_weights_only=True),
    SaveHistoryCallback()
]

# Note: building the model explicitly
vae.build((None, 128, 128, 4))

history = vae.fit(
    train_tf,
    steps_per_epoch=282, # approx 9000/32
    epochs=50,
    validation_data=val_tf,
    validation_steps=32,
    callbacks=callbacks
)


## 5. Evaluation


In [ ]:
# Load history and plot
hist_path = os.path.join(MODEL_DIR, "history.json")
if os.path.exists(hist_path):
    with open(hist_path, "r") as f:
        hist_data = json.load(f)

    plt.figure(figsize=(12, 5))
    plt.plot(hist_data.get('loss', []), label='Train Total Loss')
    plt.plot(hist_data.get('val_loss', []), label='Val Total Loss')
    plt.plot(hist_data.get('reconstruction_loss', []), label='Train Recon Loss')
    plt.plot(hist_data.get('val_reconstruction_loss', []), label='Val Recon Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('VAE Training Curves')
    plt.grid()
    plt.show()


In [ ]:
# Evaluate on Test Set
test_reconstructions = vae.predict(test_tf)

test_mse = np.mean(np.square(test_originals - test_reconstructions))
print(f"Test MSE (pixel-level): {test_mse:.6f}")

psnr_values = tf.image.psnr(test_originals, test_reconstructions, max_val=1.0).numpy()
print(f"Mean Test PSNR: {np.mean(psnr_values):.2f} dB")


In [ ]:
# Visual Reconstruction Comparison
random_indices = random.sample(range(len(test_originals)), 3)

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for i, idx in enumerate(random_indices):
    axes[0, i].imshow(test_originals[idx])
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")
    axes[1, i].imshow(test_reconstructions[idx])
    axes[1, i].set_title("VAE Reconstructed")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()


## 6. Generation


In [ ]:
# Generate from Random Noise
z_sample = np.random.normal(size=(5, latent_dim))
generated_images = decoder.predict(z_sample)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i in range(5):
    axes[i].imshow(generated_images[i])
    axes[i].set_title("Random Sample")
    axes[i].axis("off")

plt.tight_layout()
plt.show()
